# 🚀 GIAI ĐOẠN 5 — KIẾN TRÚC STAIR5-v7 / UCR-D: UPDATE-COMPATIBLE RETENTION WITH DOSE CONTROL
### 🏆 Kaggle / Colab ML Engineering Pipeline — Multimodal Recommendation with Update-Compatible Retention for Optimizer-Side Item Smoothing

> **Đề tài:** Recommender Systems using Graph Representation: Multi-modal  
> **Khóa luận tốt nghiệp:** Khóa 2021–2025 — Khoa Công nghệ Thông tin, Trường Đại học Khoa học Tự nhiên, ĐHQG-HCM  
> **Sinh viên thực hiện:** Lê Hà Thanh Chương (23120195) & Bùi Trung Hiếu (23120257)  
> **Giảng viên hướng dẫn:** TS. Nguyễn Ngọc Thảo  
> **Kiến trúc đề xuất:** **STAIR5-v7 / UCR-D** (*Update-Compatible Retention with Dose Control on Item BSC Semantic Branch*)  
> **Tài liệu đặc tả phương pháp luận:** `docs/giai_doan_5/STAIR5_v7_Report.md` (Authority Dated 2026-10-07)  
> **Mã nguồn lõi:** `models/stair5_v7.py`, `models/stair5_v7_graph.py`, `models/stair5_v7_utils.py`, `optimizers/stair5_v7_smoother.py`, `main_stair5_v7.py`  
> **Tập dữ liệu benchmark:** **Amazon Sports**, **Amazon Baby**, **Amazon Electronics** (3 tập chuẩn thương mại điện tử)  
> **Chiến lược thực thi:** **3-Tier Stage Execution** (`Pha A: Sports [Pilot] ➔ Pha B: Baby [Confirmatory] ➔ Pha C: Electronics [Scale-up]`)  
> **Cổng kiểm soát bộ nhớ (Execution Gate):** Peak VRAM Allocated $< 800\text{ MiB}$ trên Amazon Electronics  

---

## 📑 1. BẢN THIẾT KẾ KIẾN TRÚC ĐỀ XUẤT: STAIR5-v7 / UCR-D

| Thành Phần Trụ Cột | Cơ Chế Kỹ Thuật STAIR5-v7 (UCR-D) | Đột Phá & Ưu Thế So Với Baseline v4 & v6 |
|:---|:---|:---|
| **1. Kế Thừa Trọn Vẹn Backbone v4** | Modality Interaction SVD Whitening, Forward Stepwise Convolution ($L=3$), BPR Loss + Heterogeneous InfoNCE NLGCL objective. | Bảo toàn nguyên vẹn độ trễ suy luận $O(1)$; khôi phục bitwise v4 khi $\rho=0$ hoặc chọn arm `V4-control` / `V7-recovery`. |
| **2. Bắt Hướng Cập Nhật Thực Tế $D_t$** | Trích xuất $D_t = \frac{m_t / (1 - \beta_1^t)}{\sqrt{v_t / (1 - \beta_2^t)} + \epsilon}$ qua detached view từ item optimizer group. | Đánh giá trực tiếp xung đột tối ưu thực tế thay vì suy đoán gián tiếp từ gradient BPR hay EMA. |
| **3. Ngưỡng Cường Độ Thích Ứng $\tau_{\mathrm{mag}}$** | $r_i = \|D_{t, i:}\|_2$, $\tau_{\mathrm{mag}} = \max\left(10^{-12}, \zeta \cdot \operatorname{median}_{r_i > 0}(r_i)\right)$ (mặc định $\zeta=0.01$). | Loại trừ nhiễu số học từ các hàng item có độ lớn cập nhật tiệm cận 0; chỉ can thiệp trên cạnh khả dụng. |
| **4. Đo Lường Đối Kháng Từng Cặp $q_{ij}$** | $v_{ij}^{(t)} = \operatorname{clip}\left(\frac{D_{t, i:}^\top D_{t, j:}}{r_i r_j}, -1.0, 1.0\right)$, $q_{ij} = \max(0, -v_{ij})$ xử lý theo chunk ($C=4096$). | Bắt trọn xung đột hình học giữa các láng giềng semantic; chunk $C=4096$ giới hạn bộ nhớ scratchpad $<3.2\text{ MiB}$. |
| **5. Hiệu Chuẩn Liều Lượng (Dose Control)** | Ramp tăng dần $\rho_t = \rho \cdot \min(1.0, \frac{e_t + p_t}{10.0})$, tải xung đột $Z_t$, suy giảm $\theta_t = \min(\theta_{\max}, \frac{\rho_t}{Z_t})$, $a_{ij} = \theta_t q_{ij}$. | Kiểm soát chính xác tổng lượng liên kết bị triệt giảm $\rho_{\mathrm{eff}} \le \rho_t$; cap $\theta_{\max}=0.25$ ngăn xóa cạnh cực đoan. |
| **6. Hoàn Trả Khối Lượng Về Đường Chéo** | $h_{t, i} = \sum_{j \ne i} A_{t, ij}$, $W_R^{(t)} = W_0 - A_t + \operatorname{diag}(h_t) \implies W_R^{(t)} \mathbf{1} = d_0$. | Bảo toàn tuyệt đối bậc đỉnh hàng $d_0$; bù trừ lượng tương tác bị triệt giảm thành self-retention, co phổ $\|S_7\|_2 \le 1.0$. |
| **7. Topology CSR Cố Định Trên GPU** | Duy trì một cấu trúc CSR thống nhất mở rộng (chứa $S_4$ + ô đường chéo); cập nhật in-place mảng `active_values`. | Loại bỏ hoàn toàn per-step COO coalescing, CPU synchronization, và rò rỉ GPU memory context. |


## Cell 1 ⚙️ Thiết lập Môi trường, Dependencies & Đồng bộ Mã Nguồn STAIR5-v7
- Đồng bộ mã nguồn từ repository (bảo toàn mã nguồn hiện hữu, không chạy hard reset).
- Cài đặt dependencies tương thích: `freerec>=0.9.7`, `torchdata==0.8.0`, `torch-geometric`, `nvidia-ml-py`, `prettytable`, `matplotlib`, `pyyaml`, `scipy`, `pandas`.
- Kiểm tra GPU VRAM và chạy script xác thực môi trường trong tiến trình con.


In [ ]:
# Cell 1: Môi trường, Dependencies & Xác thực Subprocess
import os, shutil, subprocess, sys, glob, torch
from pathlib import Path

STAIR_DIR = '/kaggle/working/STAIR-Enhanced'
os.chdir('/kaggle/working') if os.path.exists('/kaggle/working') else None

# 1. Đồng bộ repository STAIR-Enhanced (Non-destructive)
if os.path.exists(STAIR_DIR):
    print("Thư mục STAIR-Enhanced đã tồn tại. Đang kiểm tra mã nguồn STAIR5-v7...")
    try:
        subprocess.run(['git', '-C', STAIR_DIR, 'fetch', 'origin', 'main'], check=False, capture_output=True)
        chk = subprocess.run(['git', '-C', STAIR_DIR, 'ls-tree', 'origin/main', 'models/stair5_v7.py'], capture_output=True, text=True)
        if 'models/stair5_v7.py' in chk.stdout:
            subprocess.run(['git', '-C', STAIR_DIR, 'checkout', 'main'], check=False, capture_output=True)
            subprocess.run(['git', '-C', STAIR_DIR, 'pull', 'origin', 'main'], check=False, capture_output=True)
            print("✅ Đã kiểm tra và đồng bộ cập nhật mới nhất từ origin/main.")
        else:
            print("ℹ️ Giữ nguyên mã nguồn hiện hữu trên đĩa.")
    except Exception as e:
        print(f"⚠️ Cảnh báo git ({e}). Tiếp tục dùng mã nguồn hiện hữu trên đĩa.")

if not os.path.exists(STAIR_DIR) and os.path.exists('/kaggle/working'):
    print("Cloning STAIR-Enhanced repository (branch main)...")
    try:
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ThanhChuong12/STAIR-Enhanced.git', STAIR_DIR], check=True)
    except Exception as e:
        print(f"⚠️ Git clone gặp cảnh báo: {e}")

# Fallback từ Kaggle Input nếu chưa có models/stair5_v7.py
if not os.path.exists(os.path.join(STAIR_DIR, 'models', 'stair5_v7.py')):
    input_matches = glob.glob('/kaggle/input/**/models/stair5_v7.py', recursive=True)
    if input_matches:
        src_repo = os.path.dirname(os.path.dirname(input_matches[0]))
        print(f"📦 Phát hiện mã nguồn STAIR5-v7 từ Kaggle Input: {src_repo} -> Sao chép vào {STAIR_DIR}...")
        os.makedirs(STAIR_DIR, exist_ok=True)
        shutil.copytree(src_repo, STAIR_DIR, dirs_exist_ok=True)

active_dir = STAIR_DIR if os.path.exists(STAIR_DIR) else os.path.abspath('.')
for p in [active_dir, STAIR_DIR, '/kaggle/working', '.']:
    if p and os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

if os.path.exists(STAIR_DIR):
    os.chdir(STAIR_DIR)

sub_env = os.environ.copy()
sub_env['PYTHONPATH'] = os.pathsep.join([p for p in [active_dir, os.environ.get('PYTHONPATH', '')] if p])
sub_env['PYTHONWARNINGS'] = 'ignore'
sub_env.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
os.environ['PYTHONPATH'] = sub_env['PYTHONPATH']
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

# Xóa cache module v7 cũ trong kernel
for mod_name in list(sys.modules.keys()):
    if any(k in mod_name for k in ['stair5_v7', 'models.stair5_v7', 'optimizers.stair5_v7_smoother']):
        sys.modules.pop(mod_name, None)

# 2. Cài đặt dependencies chuẩn xác
print("Cài đặt dependencies tương thích (freerec, torchdata, torch-geometric, pyyaml, prettytable)...")
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'pynvml', 'torchdata'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'torchdata==0.8.0'], check=False)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'freerec>=0.9.7', 'torch-geometric', 'nvidia-ml-py', 'prettytable', 'matplotlib', 'pyyaml', 'scipy', 'pandas'
], check=False)

# 3. Patch tương thích nội bộ PyTorch Dynamo / FreeRec
try:
    import models.freerec_compat
except Exception:
    try:
        import freerec_compat
    except Exception:
        pass

# 4. Xác minh môi trường trong tiến trình con
verify_script = '''
import torch, sys
print("=== SUBPROCESS ENVIRONMENT VERIFICATION ===")
print("Python Executable :", sys.executable)
print("Python Version    :", sys.version.split()[0])
print("PyTorch Version   :", torch.__version__)
print("CUDA Available    :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name       :", torch.cuda.get_device_name(0))
    print("Allocated VRAM    : {:.2f} MB".format(torch.cuda.memory_allocated() / (1024**2)))
try:
    import freerec
    print("FreeRec Version   :", freerec.__version__)
except Exception as e:
    print("FreeRec Import    : FAILED ({})".format(e))
try:
    from models.stair5_v7 import STAIR5_v7_Model, ARMS_V7
    print("STAIR5-v7 Arms    :", list(ARMS_V7))
    print("STAIR5-v7 Core    : VERIFIED OK")
except Exception as e:
    print("STAIR5-v7 Core    : FAILED ({})".format(e))
print("===========================================")
'''
res = subprocess.run([sys.executable, '-c', verify_script], capture_output=True, text=True, env=sub_env)
print(res.stdout)
if res.stderr and "warning" not in res.stderr.lower():
    print("Stderr:", res.stderr)
print("✅ Khởi tạo môi trường STAIR5-v7 hoàn tất thành công.")


## Cell 2 📂 Chuẩn bị Dữ liệu từ Kaggle Input (Multi-Bridge sang /kaggle/data & Processed)
- Tự động dò quét toàn bộ kho dữ liệu Kaggle Input (`/kaggle/input`) để tìm kiếm 3 tập benchmark: **Amazon Sports**, **Amazon Baby**, **Amazon Electronics**.
- Thiết lập liên kết symlink/copy đa hướng sang `/kaggle/data`, `/kaggle/data/Processed`, `STAIR-Enhanced/data` để FreeRec luôn tìm thấy dữ liệu.


In [ ]:
# Cell 2: Chuẩn bị dữ liệu từ Kaggle Input sang /kaggle/data & Processed
import os, shutil, glob

DATA_ROOT = '/kaggle/data'
PROCESSED_ROOT = os.path.join(DATA_ROOT, 'Processed')
LOCAL_DATA = '/kaggle/working/STAIR-Enhanced/data'
LOCAL_PROCESSED = os.path.join(LOCAL_DATA, 'Processed')

for d in [DATA_ROOT, PROCESSED_ROOT, LOCAL_DATA, LOCAL_PROCESSED]:
    os.makedirs(d, exist_ok=True)

TARGET_DATASETS = {
    'sports':      ('Amazon2014Sports_550_MMRec', ['sport', 'sports', 'amazon2014sports']),
    'baby':        ('Amazon2014Baby_550_MMRec', ['baby', 'amazon2014baby']),
    'electronics': ('Amazon2014Electronics_550_MMRec', ['electronic', 'electronics', 'amazon2014electronics']),
}

REQUIRED_EXTENSIONS = ('.npy', '.pkl', '.txt', '.inter', '.item', '.pt', '.csv', '.yaml')

def bridge_directories(src_dir, target_folder):
    destinations = [
        os.path.join(DATA_ROOT, target_folder),
        os.path.join(PROCESSED_ROOT, target_folder),
        os.path.join(LOCAL_DATA, target_folder),
        os.path.join(LOCAL_PROCESSED, target_folder),
    ]
    for dst in destinations:
        if os.path.abspath(src_dir) == os.path.abspath(dst):
            continue
        os.makedirs(dst, exist_ok=True)
        for item in os.listdir(src_dir):
            s_item = os.path.join(src_dir, item)
            d_item = os.path.join(dst, item)
            if os.path.isdir(s_item):
                if not os.path.exists(d_item):
                    try:
                        os.symlink(s_item, d_item)
                    except Exception:
                        shutil.copytree(s_item, d_item, dirs_exist_ok=True)
            else:
                if not os.path.exists(d_item) or os.path.getsize(d_item) == 0:
                    try:
                        os.symlink(s_item, d_item)
                    except Exception:
                        shutil.copy2(s_item, d_item)

print("=" * 80)
print("TIẾN TRÌNH DÒ QUÉT & LIÊN KẾT DỮ LIỆU TỰ ĐỘNG CHO STAIR5-v7 (UCR-D):")
prepared_data = set()
search_bases = ['/kaggle/input', '.', '..', '/kaggle/working']

for key, (folder_name, aliases) in TARGET_DATASETS.items():
    found_src = None
    for base in search_bases:
        if not os.path.exists(base):
            continue
        for root, dirs, files in os.walk(base):
            bname = os.path.basename(root).lower()
            if bname == folder_name.lower() or any(alias in bname for alias in aliases):
                has_req = any(f.endswith(REQUIRED_EXTENSIONS) for f in files)
                if has_req:
                    found_src = root
                    break
        if found_src:
            break

    if found_src:
        bridge_directories(found_src, folder_name)
        prepared_data.add(key)
        item_count = len(os.listdir(found_src))
        print(f"  ✅ [SẴN SÀNG] {key.upper():12s} -> Nguồn: {found_src} ({item_count} files)")
    else:
        print(f"  ⚠️ [CHƯA THẤY] {key.upper():12s} -> Sẽ nạp tự động qua kịch bản hoặc config.")
print("=" * 80)


## Cell 3 🧪 Bộ Kiểm Thử Độc Lập Toàn Diện Cho STAIR5-v7 UCR-D (15 Unit Tests)
- Thực thi nghiệm thu toán học và pipeline tích hợp trước khi khởi động huấn luyện.
- Kiểm tra tính bảo toàn bậc đỉnh hàng $\sum_j W_{R, ij} = d_{0, i}$, tính co phổ $\|S_7\|_2 \le 1.0$, tính triệt tiêu của phép co giãn hằng số, khôi phục bitwise v4 khi $\rho=0$, sự cô lập của optimizer parameter groups, và vòng đời snapshot GPU.
- **Fail-fast gate:** Nếu bất kỳ unit test nào thất bại, dừng ngay lập tức.


In [ ]:
# Cell 3: Chạy Bộ Kiểm Thử Độc Lập Toàn Diện Cho STAIR5-v7 (15 Unit Tests)
import os, sys, subprocess

print("=" * 80)
print("🚀 CHẠY BỘ KIỂM THỬ ĐỘC LẬP TOÀN DIỆN CHO STAIR5-v7 UCR-D (15 UNIT TESTS)...")
print("=" * 80)

test_cmd = [
    sys.executable, '-m', 'pytest',
    'tests/test_stair5_v7_graph.py',
    'tests/test_stair5_v7_pipeline.py',
    '-v'
]

sub_env = os.environ.copy()
sub_env['PYTHONPATH'] = os.pathsep.join([p for p in ['.', os.environ.get('PYTHONPATH', '')] if p])
test_cwd = '/kaggle/working/STAIR-Enhanced' if os.path.exists('/kaggle/working/STAIR-Enhanced') else '.'

res = subprocess.run(test_cmd, capture_output=True, text=True, env=sub_env, cwd=test_cwd)
print(res.stdout)
if res.returncode == 0:
    print("🎉 TẤT CẢ 15/15 UNIT TESTS CỦA STAIR5-v7 ĐÃ VƯỢT QUA XUẤT SẮC! SẴN SÀNG HUẤN LUYỆN.")
else:
    if res.stderr:
        print("STDERR:\n", res.stderr)
    raise RuntimeError(f"Kiểm thử tiền trạm thất bại với mã lỗi {res.returncode}; dừng huấn luyện để bảo vệ tài nguyên.")
print("=" * 80)


## Cell 4 🛠️ Telemetry Engine: Training Runner, GPU Hardware Profiler & Visualizer
- Quản lý tiến trình huấn luyện `main_stair5_v7.py` qua tiến trình con độc lập (`subprocess.Popen`), stream stdout thời gian thực.
- Giám sát VRAM phần cứng (`nvidia-ml-py` / NVML) và trích xuất `training_telemetry.jsonl`.
- Trích xuất tự động `selected_test_metrics.json` từ checkpoint được chọn theo validation NDCG@20.


In [ ]:
# Cell 4: Telemetry Engine — Training Runner, Hardware Profiler & Visualization
import os, sys, time, re, json, threading, subprocess
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

sys.stdout.reconfigure(encoding='utf-8') if hasattr(sys.stdout, 'reconfigure') else None

TRACKED_METRICS = ['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']

# Bảng đối chuẩn đa thế hệ lịch sử trên cả 3 tập dữ liệu
BASELINE_REF = {
    'sports':      {'Recall@10': 0.0743, 'Recall@20': 0.1111, 'NDCG@10': 0.0405, 'NDCG@20': 0.0500},
    'baby':        {'Recall@10': 0.0674, 'Recall@20': 0.1042, 'NDCG@10': 0.0359, 'NDCG@20': 0.0454},
    'electronics': {'Recall@10': 0.0442, 'Recall@20': 0.0665, 'NDCG@10': 0.0246, 'NDCG@20': 0.0303},
}

V4_REF = {
    'sports':      {'Recall@10': 0.0765, 'Recall@20': 0.1143, 'NDCG@10': 0.0419, 'NDCG@20': 0.0517},
    'baby':        {'Recall@10': 0.0678, 'Recall@20': 0.1056, 'NDCG@10': 0.0364, 'NDCG@20': 0.0461},
    'electronics': {'Recall@10': 0.0458, 'Recall@20': 0.0678, 'NDCG@10': 0.0260, 'NDCG@20': 0.0316},
}

V6_REF = {
    'sports':      {'Recall@10': 0.0765, 'Recall@20': 0.1143, 'NDCG@10': 0.0419, 'NDCG@20': 0.0517, 'epoch': 485},
    'baby':        {'Recall@10': 0.0677, 'Recall@20': 0.1056, 'NDCG@10': 0.0364, 'NDCG@20': 0.0461, 'epoch': 480},
    'electronics': {'Recall@10': 0.0458, 'Recall@20': 0.0681, 'NDCG@10': 0.0259, 'NDCG@20': 0.0317, 'epoch': 440},
}

DATASET_PROFILES = {
    'sports': {
        'name':        'Amazon Sports',
        'domain':      'E-commerce (Visual + Textual)',
        'scale':       '35,598 Users | 18,357 Items | 296K Interactions',
        'color':       '#ff7f0e',
        'approx_mins': 42.0,
    },
    'baby': {
        'name':        'Amazon Baby',
        'domain':      'E-commerce (Visual + Textual)',
        'scale':       '19,445 Users | 7,050 Items | 160K Interactions',
        'color':       '#1f77b4',
        'approx_mins': 20.0,
    },
    'electronics': {
        'name':        'Amazon Electronics',
        'domain':      'E-commerce (Visual + Textual)',
        'scale':       '192,403 Users | 63,001 Items | 1.69M Interactions',
        'color':       '#2ca02c',
        'approx_mins': 345.0,
    },
}

vram_profile = {}

def vram_monitor(key, stop_evt, interval=2.0):
    '''Background thread tracking host GPU allocated memory via NVML.'''
    try:
        import pynvml
        pynvml.nvmlInit()
        h = pynvml.nvmlDeviceGetHandleByIndex(0)
        records = []
        while not stop_evt.is_set():
            mem = pynvml.nvmlDeviceGetMemoryInfo(h)
            records.append(mem.used / (1024**2))
            time.sleep(interval)
        pynvml.nvmlShutdown()
        vram_profile[key] = records
    except Exception:
        vram_profile[key] = []

def resolve_path(p):
    if os.path.exists(p):
        return p
    base = os.path.basename(p)
    candidates = [
        p,
        os.path.join('/kaggle/working/logs/stair5_v7', base),
        os.path.join('/kaggle/working/logs/GD5', base),
        os.path.join('logs/stair5_v7', base),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    return p

def parse_telemetry_jsonl(artifact_dir):
    p = Path(artifact_dir) / "training_telemetry.jsonl"
    if not p.is_file():
        return []
    records, seen = [], set()
    with open(p, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                data = json.loads(line)
                ep = data.get("epoch")
                if ep not in seen:
                    seen.add(ep)
                    records.append(data)
            except Exception:
                continue
    return records

def extract_best_validation(log_path):
    log_path = resolve_path(log_path)
    if not os.path.exists(log_path):
        return None, {}
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    matches = re.findall(r'VALID\s+@Epoch:\s*(\d+).*?NDCG@20\s+Avg:\s*([0-9.]+)', content, re.DOTALL)
    if not matches:
        return None, {}
    best_ep, best_val = -1, -1.0
    for ep_str, val_str in matches:
        ep, val = int(ep_str), float(val_str)
        if val > best_val:
            best_val = val
            best_ep = ep
    return best_ep, {'NDCG@20': best_val}

def extract_test_metrics(log_path, artifact_dir=None):
    if artifact_dir:
        m_file = Path(artifact_dir) / "selected_test_metrics.json"
        if m_file.is_file():
            try:
                payload = json.loads(m_file.read_text(encoding="utf-8"))
                raw_m = payload.get("metrics", {})
                norm_m = {}
                for k, v in raw_m.items():
                    norm_k = k
                    for tm in TRACKED_METRICS:
                        if k.lower() == tm.lower():
                            norm_k = tm
                            break
                    norm_m[norm_k] = v
                return payload.get("epoch"), norm_m
            except Exception:
                pass
    log_path = resolve_path(log_path)
    if not os.path.exists(log_path):
        return None, {}
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    best_matches = list(re.finditer(r'\[Coach\]\s*>>>\s*Load best model\s*@Epoch:\s*(\d+)', content))
    best_match = best_matches[-1] if best_matches else None
    if not best_match:
        return None, {}
    content_after = content[best_match.end():]
    test_match = re.search(r'TEST\s+@Epoch:\s*(\d+)(.*?)(?:=================|$)', content_after, re.DOTALL)
    if not test_match:
        return None, {}
    test_epoch = int(test_match.group(1))
    test_snippet = test_match.group(2)
    test_metrics = {}
    for metric in TRACKED_METRICS:
        m = re.search(rf'{metric}(?:\s*Avg)?\s*[:\s]+\s*([0-9.]+)', test_snippet, re.IGNORECASE)
        if m:
            test_metrics[metric] = float(m.group(1))
    return test_epoch, test_metrics

def parse_training_losses(log_path, telemetry=None):
    if telemetry and len(telemetry) > 0:
        bpr_losses = [(r['epoch'], float(r['bpr_loss'])) for r in telemetry if 'bpr_loss' in r]
        cl_losses = [(r['epoch'], float(r['cl_loss'])) for r in telemetry if 'cl_loss' in r]
        total_losses = [(r['epoch'], float(r['total_loss'])) for r in telemetry if 'total_loss' in r]
        return bpr_losses, cl_losses, total_losses
    log_path = resolve_path(log_path)
    bpr_losses, cl_losses, total_losses = [], [], []
    if os.path.exists(log_path):
        with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
        matches = re.findall(r'LOSS\s+Avg:\s*([0-9.]+)', content)
        for i, val in enumerate(matches):
            total_losses.append((i + 1, float(val)))
    return bpr_losses, cl_losses, total_losses

def parse_valid_metric(log_path, metric='NDCG@20'):
    log_path = resolve_path(log_path)
    if not os.path.exists(log_path):
        return []
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    pattern = rf'VALID\s+@Epoch:\s*(\d+).*?{metric}\s+Avg:\s*([0-9.]+)'
    matches = re.findall(pattern, content, re.IGNORECASE)
    return [(int(ep), float(v)) for ep, v in matches]

def run_training_stair5_v7(key, config_yaml=None, data_root='/kaggle/data', log_path=None, artifact_dir=None, **kwargs):
    '''Executes STAIR5-v7 UCR-D training with real-time streaming and fail-fast handling.'''
    cfg_dict = STAIR5_V7_CONFIGS.get(key, {}).copy()
    cfg_dict.update(kwargs)

    config_yaml = config_yaml or cfg_dict.get('yaml')
    label = f"{key}_{cfg_dict.get('v7_arm', 'UCR-D')}_seed{cfg_dict.get('seed', 1)}"
    log_path = log_path or str(Path(cfg_dict['log']).parent / (label + '.log'))
    artifact_dir = artifact_dir or str(Path(cfg_dict['artifact_dir']).parent / label)
    resuming = bool(cfg_dict.get('resume_from'))
    if not resuming and (Path(log_path).exists() or Path(artifact_dir).exists()):
        print(f"⚠️ Thông báo: Ghi đè hoặc tái lập kịch bản run cho {label}.")
        if Path(artifact_dir).exists():
            for stale in ('manifest.json', 'training_checkpoint.pt', 'training_telemetry.jsonl', 'best_model.pt', 'last_model.pt', 'selected_test_metrics.json'):
                sp_file = Path(artifact_dir) / stale
                if sp_file.exists():
                    try:
                        sp_file.unlink()
                    except OSError:
                        pass
    STAIR5_V7_CONFIGS[key].update(log=log_path, artifact_dir=artifact_dir, v7_arm=cfg_dict.get('v7_arm', 'UCR-D'))

    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    os.makedirs(artifact_dir, exist_ok=True)

    print('=' * 80)
    print(f"🚀 KHỞI ĐỘNG TIẾN TRÌNH HUẤN LUYỆN STAIR5-v7 / UCR-D: {key.upper()}")
    print(f"  * Dataset Key         : {key}")
    print(f"  * YAML Configuration  : {config_yaml}")
    print(f"  * Log Path            : {log_path}")
    print(f"  * Artifact Directory  : {artifact_dir}")
    print(f"  * V7 Arm              : {cfg_dict.get('v7_arm', 'UCR-D')}")
    print(f"  * Target Dose (ρ)     : {cfg_dict.get('v7_rho', 0.01)}")
    print(f"  * Attenuation Cap (θ) : {cfg_dict.get('v7_theta_max', 0.25)}")
    print(f"  * Magnitude Mult (ζ)  : {cfg_dict.get('v7_zeta', 0.01)}")
    print(f"  * Operator Blend (η)  : {cfg_dict.get('eta', 0.1)}")
    print(f"  * Seed                : {cfg_dict.get('seed', 1)}")
    print('=' * 80)

    stop_evt = threading.Event()
    th = threading.Thread(target=vram_monitor, args=(key, stop_evt), daemon=True)
    th.start()

    t0 = time.time()
    runner_py = '/kaggle/working/STAIR-Enhanced/main_stair5_v7.py'
    if not os.path.exists(runner_py):
        runner_py = 'main_stair5_v7.py'

    if not os.path.exists(config_yaml):
        cand_y = os.path.join('configs', os.path.basename(config_yaml))
        if os.path.exists(cand_y):
            config_yaml = cand_y

    device_str = str(cfg_dict.get('device', '0' if torch.cuda.is_available() else 'cpu'))
    cmd = [
        sys.executable, '-u', str(runner_py),
        '--config', str(config_yaml),
        '--root', str(data_root),
        '--device', str(device_str),
        '--ranking', 'full',
        '--artifact-dir', str(artifact_dir),
    ]

    defaults = {
        'v7_arm': 'UCR-D', 'v7_rho': 0.01, 'v7_theta_max': 0.25, 'v7_zeta': 0.01,
        'v7_pair_chunk_size': 4096, 'v7_warmup_epochs': 10.0, 'v7_shuffled_seed': 1,
        'seed': 1, 'eval_freq': 5, 'eta': 0.1, 'k_cf': 5, 'c_min': 2, 't_shrinkage': 5.0,
        'cf_block_size': 64, 'cf_memory_budget_mib': 128,
        'lambda_nlgcl': 0.01, 'nlgcl_tau': 0.2, 'nlgcl_G': 1, 'nlgcl_alpha': 0.5, 'cl_chunk_size': 1024,
        'knn_chunk_size': 1024, 'knn_device': 'auto',
        'graph_cache_dir': '/kaggle/working/runs/stair5_v7/graph_cache',
    }
    settings = {**defaults, **cfg_dict}
    for k_item in defaults:
        cmd += ['--' + k_item.replace('_', '-'), str(settings[k_item])]
    for k_item in ('epochs', 'batch_size', 'gamma', 'lr', 'weight_decay', 'num_workers'):
        if k_item in cfg_dict:
            cmd += ['--' + k_item.replace('_', '-'), str(cfg_dict[k_item])]
    if cfg_dict.get('resume_from'):
        cmd += ['--resume-from', str(cfg_dict['resume_from'])]

    print('Command:', ' '.join(cmd))
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([p for p in ['.', active_dir, os.environ.get('PYTHONPATH', '')] if p])
    env['PYTHONUTF8'] = '1'

    try:
        with open(log_path, 'a' if resuming else 'w', encoding='utf-8') as lf:
            proc = subprocess.Popen(
                cmd,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                encoding='utf-8', errors='replace',
                bufsize=1,
                env=env,
            )
            for line in proc.stdout:
                sys.stdout.write(line)
                sys.stdout.flush()
                lf.write(line)
                lf.flush()
            proc.wait()
            ret_code = proc.returncode
    finally:
        stop_evt.set()
        th.join(timeout=3.0)

    elapsed = time.time() - t0
    print(f"⏱️ Thời gian thực thi ({key.upper()}): {elapsed/60:.2f} phút | Trạng thái exit code: {ret_code}")
    if ret_code != 0:
        raise RuntimeError(f"Tiến trình huấn luyện {key.upper()} thất bại với mã lỗi {ret_code}. Kiểm tra log: {log_path}")
    return elapsed


## Cell 5 📋 Cấu hình Siêu tham số STAIR5-v7 UCR-D (Dataset-Adaptive Matrix)
- Ma trận cấu hình chi tiết cho 3 tập benchmark theo đặc tả toán học trong `docs/giai_doan_5/STAIR5_v7_Report.md`:
  * **Amazon Sports**: $\rho = 0.01, \theta_{\max} = 0.25, \zeta = 0.01, \gamma = 0.2, wd = 0.1, \text{lr} = 0.001, \text{batch\_size} = 1024$.
  * **Amazon Baby**: $\rho = 0.01, \theta_{\max} = 0.25, \zeta = 0.01, \gamma = 0.1, wd = 0.3, \text{lr} = 0.001, \text{batch\_size} = 1024$.
  * **Amazon Electronics**: $\rho = 0.01, \theta_{\max} = 0.25, \zeta = 0.01, \gamma = 0.4, wd = 0.1, \text{lr} = 0.001, \text{batch\_size} = 4096$, memory gate $<800\text{ MiB}$.


In [ ]:
# Cell 5: Cấu hình Siêu tham số STAIR5-v7 UCR-D (Dataset-Adaptive Matrix)
import os, uuid
from datetime import datetime
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:6]

os.makedirs('/kaggle/working/logs/stair5_v7', exist_ok=True)
os.makedirs('/kaggle/working/reports', exist_ok=True)
os.makedirs('/kaggle/working/runs/stair5_v7/graph_cache', exist_ok=True)

STAIR5_V7_CONFIGS = {
    'sports': {
        'yaml':               '/kaggle/working/STAIR-Enhanced/configs/Amazon2014Sports_STAIR5_v7.yaml',
        'log':                f'/kaggle/working/logs/stair5_v7/{RUN_ID}/sports_stair5_v7.log',
        'artifact_dir':       f'/kaggle/working/runs/stair5_v7/{RUN_ID}/sports_UCR-D_seed1',
        'v7_arm':             'UCR-D',
        'v7_rho':             0.01,
        'v7_theta_max':       0.25,
        'v7_zeta':            0.01,
        'v7_pair_chunk_size': 4096,
        'v7_warmup_epochs':   10.0,
        'eta':                0.1,
        'k_cf':               5,
        'c_min':              2,
        't_shrinkage':        5.0,
        'cf_block_size':      64,
        'lambda_nlgcl':       0.01,
        'nlgcl_tau':          0.2,
        'nlgcl_G':            1,
        'nlgcl_alpha':        0.5,
        'cl_chunk_size':      1024,
        'seed':               1,
        'eval_freq':          5,
    },
    'baby': {
        'yaml':               '/kaggle/working/STAIR-Enhanced/configs/Amazon2014Baby_STAIR5_v7.yaml',
        'log':                f'/kaggle/working/logs/stair5_v7/{RUN_ID}/baby_stair5_v7.log',
        'artifact_dir':       f'/kaggle/working/runs/stair5_v7/{RUN_ID}/baby_UCR-D_seed1',
        'v7_arm':             'UCR-D',
        'v7_rho':             0.01,
        'v7_theta_max':       0.25,
        'v7_zeta':            0.01,
        'v7_pair_chunk_size': 4096,
        'v7_warmup_epochs':   10.0,
        'eta':                0.1,
        'k_cf':               5,
        'c_min':              2,
        't_shrinkage':        5.0,
        'cf_block_size':      64,
        'lambda_nlgcl':       0.01,
        'nlgcl_tau':          0.2,
        'nlgcl_G':            1,
        'nlgcl_alpha':        0.5,
        'cl_chunk_size':      1024,
        'seed':               1,
        'eval_freq':          5,
    },
    'electronics': {
        'yaml':               '/kaggle/working/STAIR-Enhanced/configs/Amazon2014Electronics_STAIR5_v7.yaml',
        'log':                f'/kaggle/working/logs/stair5_v7/{RUN_ID}/electronics_stair5_v7.log',
        'artifact_dir':       f'/kaggle/working/runs/stair5_v7/{RUN_ID}/electronics_UCR-D_seed1',
        'v7_arm':             'UCR-D',
        'v7_rho':             0.01,
        'v7_theta_max':       0.25,
        'v7_zeta':            0.01,
        'v7_pair_chunk_size': 4096,
        'v7_warmup_epochs':   10.0,
        'eta':                0.1,
        'k_cf':               5,
        'c_min':              2,
        't_shrinkage':        5.0,
        'cf_block_size':      64,
        'lambda_nlgcl':       0.01,
        'nlgcl_tau':          0.2,
        'nlgcl_G':            1,
        'nlgcl_alpha':        0.5,
        'cl_chunk_size':      1024,
        'seed':               1,
        'eval_freq':          5,
    },
}

print('✅ Ma trận cấu hình STAIR5-v7 UCR-D đã nạp thành công:')
for k, v in STAIR5_V7_CONFIGS.items():
    print(f"  * {k.upper():12s} -> Arm: {v['v7_arm']} | Config: {os.path.basename(v['yaml'])} | Rho: {v['v7_rho']} | ThetaMax: {v['v7_theta_max']}")


## Cell 6a 🏋️ Huấn luyện Pha A — Amazon Sports (Pilot STAIR5-v7 / UCR-D)
- Huấn luyện mô hình STAIR5-v7 trên tập **Amazon Sports** (35,598 Users, 18,357 Items).
- Ghi nhận telemetry từng epoch: BPR loss, NLGCL InfoNCE loss, VRAM phân bổ, và telemetry xung đột $Z_t, \theta_t, \rho_{\mathrm{eff}}$.


In [ ]:
# Cell 6a: Training STAIR5-v7 UCR-D on Amazon Sports (Pha A: Pilot)
sports_time = run_training_stair5_v7('sports')
ep_sports, metrics_sports = extract_test_metrics(STAIR5_V7_CONFIGS['sports']['log'], STAIR5_V7_CONFIGS['sports']['artifact_dir'])
val_ep_sports, val_m_sports = extract_best_validation(STAIR5_V7_CONFIGS['sports']['log'])

print('\n' + '=' * 80)
print('📊 KẾT QUẢ THỰC NGHIỆM PHA A — AMAZON SPORTS (STAIR5-v7 / UCR-D):')
print(f"  * Checkpoint tối ưu tại Epoch: {ep_sports} (Validation NDCG@20 = {val_m_sports.get('NDCG@20', 0.0):.4f})")
for m in TRACKED_METRICS:
    val = metrics_sports.get(m, 0.0)
    base = BASELINE_REF['sports'].get(m, 0.0)
    v4_val = V4_REF['sports'].get(m, 0.0)
    v6_val = V6_REF['sports'].get(m, 0.0)
    d_base = ((val - base) / base) * 100 if base > 0 else 0.0
    d_v4 = ((val - v4_val) / v4_val) * 100 if v4_val > 0 else 0.0
    d_v6 = ((val - v6_val) / v6_val) * 100 if v6_val > 0 else 0.0
    print(f"  * {m:10s} : {val:.4f}  (vs Baseline: {d_base:+.2f}% | vs v4: {d_v4:+.2f}% | vs v6: {d_v6:+.2f}%)")
print('=' * 80)


## Cell 6b ⚡ Biểu đồ Tiêu thụ Bộ nhớ Tensor Mô hình — Amazon Sports
- Phân tích chi tiết mức tiêu thụ VRAM thực tế (`max_memory_allocated_bytes`) trên Amazon Sports.


In [ ]:
# Cell 6b: Model Tensor VRAM Profile — Amazon Sports (Paper Standard)
sports_art = STAIR5_V7_CONFIGS['sports']['artifact_dir']
telemetry_sports = parse_telemetry_jsonl(sports_art)

plt.figure(figsize=(9, 4.5), dpi=150)
if telemetry_sports:
    eps = [r['epoch'] for r in telemetry_sports]
    vram_mb = [r.get('max_memory_allocated_bytes', 0) / (1024**2) for r in telemetry_sports]
    plt.plot(eps, vram_mb, color='#ff7f0e', lw=2.2, marker='o', ms=4, label='Peak Allocated VRAM (MB)')
    plt.title('Amazon Sports — Peak Model Tensor VRAM Profile (STAIR5-v7)', fontsize=12, fontweight='bold')
    plt.xlabel('Epoch', fontsize=10)
    plt.ylabel('Allocated VRAM (MB)', fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(frameon=True)
elif 'sports' in vram_profile and len(vram_profile['sports']) > 0:
    plt.plot(vram_profile['sports'], color='#ff7f0e', lw=1.8, label='Host NVML GPU Used (MB)')
    plt.title('Amazon Sports — Host GPU Memory Profile (NVML)', fontsize=12, fontweight='bold')
    plt.xlabel('Polling Step (~2s)', fontsize=10)
    plt.ylabel('Memory (MB)', fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(frameon=True)
else:
    plt.text(0.5, 0.5, 'Chưa có dữ liệu VRAM hoặc NVML không khả dụng', ha='center', va='center')

plt.tight_layout()
os.makedirs('/kaggle/working/reports', exist_ok=True)
plt.savefig('/kaggle/working/reports/vram_profile_sports.png')
plt.show()


## Cell 6c 📈 Động Lực Học & Quá Trình Hội Tụ — Amazon Sports
- Trực quan hóa hàm mất mát (Total loss, BPR loss) và đường cong Validation NDCG@20 so với v4 và v6.


In [ ]:
# Cell 6c: Learning Dynamics & Convergence Profiles — Amazon Sports
fig, ax1 = plt.subplots(figsize=(10, 4.8), dpi=150)

log_sports = STAIR5_V7_CONFIGS['sports']['log']
bpr_l, cl_l, tot_l = parse_training_losses(log_sports, telemetry_sports)
val_ndcg = parse_valid_metric(log_sports, 'NDCG@20')

if tot_l:
    x_tot, y_tot = zip(*tot_l)
    ax1.plot(x_tot, y_tot, color='#1f77b4', lw=2.0, label='Total Loss (BPR + λ·NLGCL)')
if bpr_l:
    x_bpr, y_bpr = zip(*bpr_l)
    ax1.plot(x_bpr, y_bpr, color='#1f77b4', lw=1.5, ls='--', alpha=0.7, label='BPR Loss')

ax1.set_xlabel('Epoch', fontsize=10)
ax1.set_ylabel('Training Loss', color='#1f77b4', fontsize=10)
ax1.tick_params(axis='y', labelcolor='#1f77b4')
ax1.grid(True, linestyle='--', alpha=0.4)

if val_ndcg:
    ax2 = ax1.twinx()
    x_val, y_val = zip(*val_ndcg)
    ax2.plot(x_val, y_val, color='#ff7f0e', lw=2.2, marker='o', ms=4, label='Validation NDCG@20')
    ax2.set_ylabel('Validation NDCG@20', color='#ff7f0e', fontsize=10)
    ax2.tick_params(axis='y', labelcolor='#ff7f0e')
    if 'sports' in BASELINE_REF:
        ax2.axhline(BASELINE_REF['sports']['NDCG@20'], color='gray', ls=':', label='Baseline STAIR')
    if 'sports' in V4_REF:
        ax2.axhline(V4_REF['sports']['NDCG@20'], color='#2ca02c', ls='--', label='GD5-v4 N-CSE')
    if 'sports' in V6_REF:
        ax2.axhline(V6_REF['sports']['NDCG@20'], color='#9467bd', ls='-.', label='GD5-v6 P-BCSR')

plt.title('Amazon Sports — Learning Dynamics (STAIR5-v7 UCR-D)', fontsize=12, fontweight='bold')
fig.tight_layout()
plt.savefig('/kaggle/working/reports/learning_curve_sports.png')
plt.show()


## Cell 6d 🎯 Phân tích Động lực Học Liều lượng & Xung đột UCR-D — Amazon Sports
- Giám sát độ khả thi của liều lượng: Biểu diễn liều lượng mục tiêu $\rho_t$ so với liều lượng thực tế đạt được $\rho_{\mathrm{eff}}$.
- Giám sát tải xung đột $Z_t$ và hệ số suy giảm $\theta_t$ theo epoch.


In [ ]:
# Cell 6d: UCR-D Dynamic Dose & Conflict Telemetry — Amazon Sports
plt.figure(figsize=(10, 4.5), dpi=150)
if telemetry_sports:
    eps = [r['epoch'] for r in telemetry_sports if 'diagnostics' in r and r['diagnostics']]
    rho_eff = [r['diagnostics'].get('rho_eff', 0.0) for r in telemetry_sports if 'diagnostics' in r and r['diagnostics']]
    rho_t = [r['diagnostics'].get('rho_t', 0.0) for r in telemetry_sports if 'diagnostics' in r and r['diagnostics']]
    Z_t = [r['diagnostics'].get('Z_t', 0.0) for r in telemetry_sports if 'diagnostics' in r and r['diagnostics']]

    if eps:
        plt.plot(eps, rho_t, color='#1f77b4', lw=2.0, ls='--', label='Target Dose ρ_t')
        plt.plot(eps, rho_eff, color='#2ca02c', lw=2.2, label='Achieved Dose ρ_eff')
        plt.plot(eps, Z_t, color='#d62728', lw=1.5, alpha=0.7, label='Conflict Load Z_t')
        plt.title('Amazon Sports — UCR-D Dose Feasibility & Conflict Trajectory', fontsize=12, fontweight='bold')
        plt.xlabel('Epoch', fontsize=10)
        plt.ylabel('Metric Value', fontsize=10)
        plt.grid(True, linestyle='--', alpha=0.5)
        plt.legend(frameon=True)
    else:
        plt.text(0.5, 0.5, 'Chưa có telemetry UCR-D (Chế độ Fast-Path hoặc Rho=0)', ha='center', va='center')
else:
    plt.text(0.5, 0.5, 'Chưa có telemetry training', ha='center', va='center')

plt.tight_layout()
plt.savefig('/kaggle/working/reports/ucr_dynamics_sports.png')
plt.show()


## Cell 7a 🏋️ Huấn luyện Pha B — Amazon Baby (Kiểm chứng An toàn STAIR5-v7 / UCR-D)
- Huấn luyện STAIR5-v7 trên tập **Amazon Baby** (19,445 Users, 7,050 Items).
- Kiểm tra tính ổn định trên tập mật độ thưa cao với siêu tham số $\gamma=0.1, wd=0.3$.


In [ ]:
# Cell 7a: Training STAIR5-v7 UCR-D on Amazon Baby (Pha B: Confirmatory)
baby_time = run_training_stair5_v7('baby')
ep_baby, metrics_baby = extract_test_metrics(STAIR5_V7_CONFIGS['baby']['log'], STAIR5_V7_CONFIGS['baby']['artifact_dir'])
val_ep_baby, val_m_baby = extract_best_validation(STAIR5_V7_CONFIGS['baby']['log'])

print('\n' + '=' * 80)
print('📊 KẾT QUẢ THỰC NGHIỆM PHA B — AMAZON BABY (STAIR5-v7 / UCR-D):')
print(f"  * Checkpoint tối ưu tại Epoch: {ep_baby} (Validation NDCG@20 = {val_m_baby.get('NDCG@20', 0.0):.4f})")
for m in TRACKED_METRICS:
    val = metrics_baby.get(m, 0.0)
    base = BASELINE_REF['baby'].get(m, 0.0)
    v4_val = V4_REF['baby'].get(m, 0.0)
    v6_val = V6_REF['baby'].get(m, 0.0)
    d_base = ((val - base) / base) * 100 if base > 0 else 0.0
    d_v4 = ((val - v4_val) / v4_val) * 100 if v4_val > 0 else 0.0
    d_v6 = ((val - v6_val) / v6_val) * 100 if v6_val > 0 else 0.0
    print(f"  * {m:10s} : {val:.4f}  (vs Baseline: {d_base:+.2f}% | vs v4: {d_v4:+.2f}% | vs v6: {d_v6:+.2f}%)")
print('=' * 80)


## Cell 7b ⚡ Biểu đồ Tiêu thụ Bộ nhớ Tensor Mô hình — Amazon Baby
- Phân tích chi tiết mức tiêu thụ VRAM thực tế trên Amazon Baby.


In [ ]:
# Cell 7b: Model Tensor VRAM Profile — Amazon Baby (Paper Standard)
baby_art = STAIR5_V7_CONFIGS['baby']['artifact_dir']
telemetry_baby = parse_telemetry_jsonl(baby_art)

plt.figure(figsize=(9, 4.5), dpi=150)
if telemetry_baby:
    eps = [r['epoch'] for r in telemetry_baby]
    vram_mb = [r.get('max_memory_allocated_bytes', 0) / (1024**2) for r in telemetry_baby]
    plt.plot(eps, vram_mb, color='#1f77b4', lw=2.2, marker='o', ms=4, label='Peak Allocated VRAM (MB)')
    plt.title('Amazon Baby — Peak Model Tensor VRAM Profile (STAIR5-v7)', fontsize=12, fontweight='bold')
    plt.xlabel('Epoch', fontsize=10)
    plt.ylabel('Allocated VRAM (MB)', fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(frameon=True)
elif 'baby' in vram_profile and len(vram_profile['baby']) > 0:
    plt.plot(vram_profile['baby'], color='#1f77b4', lw=1.8, label='Host NVML GPU Used (MB)')
    plt.title('Amazon Baby — Host GPU Memory Profile (NVML)', fontsize=12, fontweight='bold')
    plt.xlabel('Polling Step (~2s)', fontsize=10)
    plt.ylabel('Memory (MB)', fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(frameon=True)
else:
    plt.text(0.5, 0.5, 'Chưa có dữ liệu VRAM hoặc NVML không khả dụng', ha='center', va='center')

plt.tight_layout()
os.makedirs('/kaggle/working/reports', exist_ok=True)
plt.savefig('/kaggle/working/reports/vram_profile_baby.png')
plt.show()


## Cell 7c 📈 Động Lực Học & Quá Trình Hội Tụ — Amazon Baby
- Trực quan hóa đường cong hàm mất mát BPR và đường cong đánh giá Validation NDCG@20 trên Amazon Baby.


In [ ]:
# Cell 7c: Learning Dynamics & Convergence Profiles — Amazon Baby
fig, ax1 = plt.subplots(figsize=(10, 4.8), dpi=150)

log_baby = STAIR5_V7_CONFIGS['baby']['log']
bpr_b, cl_b, tot_b = parse_training_losses(log_baby, telemetry_baby)
val_ndcg_b = parse_valid_metric(log_baby, 'NDCG@20')

if tot_b:
    x_tot, y_tot = zip(*tot_b)
    ax1.plot(x_tot, y_tot, color='#1f77b4', lw=2.0, label='Total Loss (BPR + λ·NLGCL)')
if bpr_b:
    x_bpr, y_bpr = zip(*bpr_b)
    ax1.plot(x_bpr, y_bpr, color='#1f77b4', lw=1.5, ls='--', alpha=0.7, label='BPR Loss')

ax1.set_xlabel('Epoch', fontsize=10)
ax1.set_ylabel('Training Loss', color='#1f77b4', fontsize=10)
ax1.tick_params(axis='y', labelcolor='#1f77b4')
ax1.grid(True, linestyle='--', alpha=0.4)

if val_ndcg_b:
    ax2 = ax1.twinx()
    x_val, y_val = zip(*val_ndcg_b)
    ax2.plot(x_val, y_val, color='#ff7f0e', lw=2.2, marker='o', ms=4, label='Validation NDCG@20')
    ax2.set_ylabel('Validation NDCG@20', color='#ff7f0e', fontsize=10)
    ax2.tick_params(axis='y', labelcolor='#ff7f0e')
    if 'baby' in BASELINE_REF:
        ax2.axhline(BASELINE_REF['baby']['NDCG@20'], color='gray', ls=':', label='Baseline STAIR')
    if 'baby' in V4_REF:
        ax2.axhline(V4_REF['baby']['NDCG@20'], color='#2ca02c', ls='--', label='GD5-v4 N-CSE')
    if 'baby' in V6_REF:
        ax2.axhline(V6_REF['baby']['NDCG@20'], color='#9467bd', ls='-.', label='GD5-v6 P-BCSR')

plt.title('Amazon Baby — Learning Dynamics (STAIR5-v7 UCR-D)', fontsize=12, fontweight='bold')
fig.tight_layout()
plt.savefig('/kaggle/working/reports/learning_curve_baby.png')
plt.show()


## Cell 7d 🎯 Phân tích Động lực Học Liều lượng & Xung đột UCR-D — Amazon Baby
- Giám sát độ khả thi của liều lượng: $\rho_t$ vs $\rho_{\mathrm{eff}}$ và tải xung đột $Z_t$ trên Amazon Baby.


In [ ]:
# Cell 7d: UCR-D Dynamic Dose & Conflict Telemetry — Amazon Baby
plt.figure(figsize=(10, 4.5), dpi=150)
if telemetry_baby:
    eps = [r['epoch'] for r in telemetry_baby if 'diagnostics' in r and r['diagnostics']]
    rho_eff = [r['diagnostics'].get('rho_eff', 0.0) for r in telemetry_baby if 'diagnostics' in r and r['diagnostics']]
    rho_t = [r['diagnostics'].get('rho_t', 0.0) for r in telemetry_baby if 'diagnostics' in r and r['diagnostics']]
    Z_t = [r['diagnostics'].get('Z_t', 0.0) for r in telemetry_baby if 'diagnostics' in r and r['diagnostics']]

    if eps:
        plt.plot(eps, rho_t, color='#1f77b4', lw=2.0, ls='--', label='Target Dose ρ_t')
        plt.plot(eps, rho_eff, color='#2ca02c', lw=2.2, label='Achieved Dose ρ_eff')
        plt.plot(eps, Z_t, color='#d62728', lw=1.5, alpha=0.7, label='Conflict Load Z_t')
        plt.title('Amazon Baby — UCR-D Dose Feasibility & Conflict Trajectory', fontsize=12, fontweight='bold')
        plt.xlabel('Epoch', fontsize=10)
        plt.ylabel('Metric Value', fontsize=10)
        plt.grid(True, linestyle='--', alpha=0.5)
        plt.legend(frameon=True)
    else:
        plt.text(0.5, 0.5, 'Chưa có telemetry UCR-D (Chế độ Fast-Path hoặc Rho=0)', ha='center', va='center')
else:
    plt.text(0.5, 0.5, 'Chưa có telemetry training', ha='center', va='center')

plt.tight_layout()
plt.savefig('/kaggle/working/reports/ucr_dynamics_baby.png')
plt.show()


## Cell 8a 🏋️ Huấn luyện Pha C — Amazon Electronics (~1.7M Tương tác, Cổng kiểm soát $<800\text{ MiB}$)
- Kiểm chứng khả năng mở rộng (Scalability) của STAIR5-v7 trên tập quy mô lớn **Amazon Electronics** (192,403 Users, 63,001 Items, 1.69M Interactions).
- Sử dụng cấu hình chuẩn: $\gamma = 0.4, \text{batch\_size} = 4096, \text{cl\_chunk\_size} = 1024, \text{cf\_block\_size} = 64$.
- **Cổng kiểm soát bộ nhớ:** Bắt buộc VRAM phân bổ đỉnh $\le 800\text{ MiB}$.


In [ ]:
# Cell 8a: Training STAIR5-v7 UCR-D on Amazon Electronics (Pha C: Scale-up)
electronics_time = run_training_stair5_v7('electronics')
ep_elec, metrics_elec = extract_test_metrics(STAIR5_V7_CONFIGS['electronics']['log'], STAIR5_V7_CONFIGS['electronics']['artifact_dir'])
val_ep_elec, val_m_elec = extract_best_validation(STAIR5_V7_CONFIGS['electronics']['log'])

print('\n' + '=' * 80)
print('📊 KẾT QUẢ THỰC NGHIỆM PHA C — AMAZON ELECTRONICS (STAIR5-v7 / UCR-D):')
print(f"  * Checkpoint tối ưu tại Epoch: {ep_elec} (Validation NDCG@20 = {val_m_elec.get('NDCG@20', 0.0):.4f})")
for m in TRACKED_METRICS:
    val = metrics_elec.get(m, 0.0)
    base = BASELINE_REF['electronics'].get(m, 0.0)
    v4_val = V4_REF['electronics'].get(m, 0.0)
    v6_val = V6_REF['electronics'].get(m, 0.0)
    d_base = ((val - base) / base) * 100 if base > 0 else 0.0
    d_v4 = ((val - v4_val) / v4_val) * 100 if v4_val > 0 else 0.0
    d_v6 = ((val - v6_val) / v6_val) * 100 if v6_val > 0 else 0.0
    print(f"  * {m:10s} : {val:.4f}  (vs Baseline: {d_base:+.2f}% | vs v4: {d_v4:+.2f}% | vs v6: {d_v6:+.2f}%)")
print('=' * 80)


## Cell 8b ⚡ Biểu đồ Tiêu thụ Bộ nhớ Tensor Mô hình — Amazon Electronics (Kiểm chứng Cổng $<800\text{ MiB}$)
- Phân tích chi tiết mức tiêu thụ VRAM thực tế trên Amazon Electronics.
- Vẽ đường kiểm soát ngưỡng $800\text{ MiB}$ để minh chứng thỏa mãn điều kiện phần cứng.


In [ ]:
# Cell 8b: Model Tensor VRAM Profile — Amazon Electronics (Memory Gate Verification)
elec_art = STAIR5_V7_CONFIGS['electronics']['artifact_dir']
telemetry_elec = parse_telemetry_jsonl(elec_art)

plt.figure(figsize=(9, 4.5), dpi=150)
if telemetry_elec:
    eps = [r['epoch'] for r in telemetry_elec]
    vram_mb = [r.get('max_memory_allocated_bytes', 0) / (1024**2) for r in telemetry_elec]
    peak_val = max(vram_mb)
    plt.plot(eps, vram_mb, color='#2ca02c', lw=2.0, label=f'Peak Allocated VRAM (Max: {peak_val:.1f} MB)')
    plt.axhline(800.0, color='red', ls='--', lw=1.8, label='Execution Gate Threshold (800 MB)')
    plt.title('Amazon Electronics — Peak Model Tensor VRAM Profile (STAIR5-v7 Gate Check)', fontsize=12, fontweight='bold')
    plt.xlabel('Epoch', fontsize=10)
    plt.ylabel('Allocated VRAM (MB)', fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(frameon=True)
elif 'electronics' in vram_profile and len(vram_profile['electronics']) > 0:
    plt.plot(vram_profile['electronics'], color='#2ca02c', lw=1.8, label='Host NVML GPU Used (MB)')
    plt.axhline(800.0, color='red', ls='--', lw=1.8, label='Execution Gate Threshold (800 MB)')
    plt.title('Amazon Electronics — Host GPU Memory Profile (NVML)', fontsize=12, fontweight='bold')
    plt.xlabel('Polling Step (~2s)', fontsize=10)
    plt.ylabel('Memory (MB)', fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(frameon=True)
else:
    plt.text(0.5, 0.5, 'Chưa có dữ liệu VRAM hoặc NVML không khả dụng', ha='center', va='center')

plt.tight_layout()
os.makedirs('/kaggle/working/reports', exist_ok=True)
plt.savefig('/kaggle/working/reports/vram_profile_electronics.png')
plt.show()


## Cell 8c 📈 Động Lực Học & Quá Trình Hội Tụ — Amazon Electronics
- Trực quan hóa đường cong hàm mất mát BPR và đường cong đánh giá Validation NDCG@20 trên Amazon Electronics.


In [ ]:
# Cell 8c: Learning Dynamics & Convergence Profiles — Amazon Electronics
fig, ax1 = plt.subplots(figsize=(10, 4.8), dpi=150)

log_elec = STAIR5_V7_CONFIGS['electronics']['log']
bpr_e, cl_e, tot_e = parse_training_losses(log_elec, telemetry_elec)
val_ndcg_e = parse_valid_metric(log_elec, 'NDCG@20')

if tot_e:
    x_tot, y_tot = zip(*tot_e)
    ax1.plot(x_tot, y_tot, color='#1f77b4', lw=2.0, label='Total Loss (BPR + λ·NLGCL)')
if bpr_e:
    x_bpr, y_bpr = zip(*bpr_e)
    ax1.plot(x_bpr, y_bpr, color='#1f77b4', lw=1.5, ls='--', alpha=0.7, label='BPR Loss')

ax1.set_xlabel('Epoch', fontsize=10)
ax1.set_ylabel('Training Loss', color='#1f77b4', fontsize=10)
ax1.tick_params(axis='y', labelcolor='#1f77b4')
ax1.grid(True, linestyle='--', alpha=0.4)

if val_ndcg_e:
    ax2 = ax1.twinx()
    x_val, y_val = zip(*val_ndcg_e)
    ax2.plot(x_val, y_val, color='#2ca02c', lw=2.2, marker='o', ms=4, label='Validation NDCG@20')
    ax2.set_ylabel('Validation NDCG@20', color='#2ca02c', fontsize=10)
    ax2.tick_params(axis='y', labelcolor='#2ca02c')
    if 'electronics' in BASELINE_REF:
        ax2.axhline(BASELINE_REF['electronics']['NDCG@20'], color='gray', ls=':', label='Baseline STAIR')
    if 'electronics' in V4_REF:
        ax2.axhline(V4_REF['electronics']['NDCG@20'], color='#9467bd', ls='--', label='GD5-v4 N-CSE')
    if 'electronics' in V6_REF:
        ax2.axhline(V6_REF['electronics']['NDCG@20'], color='#ff7f0e', ls='-.', label='GD5-v6 P-BCSR')

plt.title('Amazon Electronics — Learning Dynamics (STAIR5-v7 UCR-D)', fontsize=12, fontweight='bold')
fig.tight_layout()
plt.savefig('/kaggle/working/reports/learning_curve_electronics.png')
plt.show()


## Cell 8d 🎯 Phân tích Động lực Học Liều lượng & Xung đột UCR-D — Amazon Electronics
- Giám sát độ khả thi của liều lượng: $\rho_t$ vs $\rho_{\mathrm{eff}}$ và tải xung đột $Z_t$ trên Amazon Electronics.


In [ ]:
# Cell 8d: UCR-D Dynamic Dose & Conflict Telemetry — Amazon Electronics
plt.figure(figsize=(10, 4.5), dpi=150)
if telemetry_elec:
    eps = [r['epoch'] for r in telemetry_elec if 'diagnostics' in r and r['diagnostics']]
    rho_eff = [r['diagnostics'].get('rho_eff', 0.0) for r in telemetry_elec if 'diagnostics' in r and r['diagnostics']]
    rho_t = [r['diagnostics'].get('rho_t', 0.0) for r in telemetry_elec if 'diagnostics' in r and r['diagnostics']]
    Z_t = [r['diagnostics'].get('Z_t', 0.0) for r in telemetry_elec if 'diagnostics' in r and r['diagnostics']]

    if eps:
        plt.plot(eps, rho_t, color='#1f77b4', lw=2.0, ls='--', label='Target Dose ρ_t')
        plt.plot(eps, rho_eff, color='#2ca02c', lw=2.2, label='Achieved Dose ρ_eff')
        plt.plot(eps, Z_t, color='#d62728', lw=1.5, alpha=0.7, label='Conflict Load Z_t')
        plt.title('Amazon Electronics — UCR-D Dose Feasibility & Conflict Trajectory', fontsize=12, fontweight='bold')
        plt.xlabel('Epoch', fontsize=10)
        plt.ylabel('Metric Value', fontsize=10)
        plt.grid(True, linestyle='--', alpha=0.5)
        plt.legend(frameon=True)
    else:
        plt.text(0.5, 0.5, 'Chưa có telemetry UCR-D (Chế độ Fast-Path hoặc Rho=0)', ha='center', va='center')
else:
    plt.text(0.5, 0.5, 'Chưa có telemetry training', ha='center', va='center')

plt.tight_layout()
plt.savefig('/kaggle/working/reports/ucr_dynamics_electronics.png')
plt.show()


## Cell 9 📊 Bảng Đối chuẩn Đa thế hệ Toàn diện (Baseline vs v4 vs v6 vs v7)
- Bảng tổng hợp đối sánh toàn diện 4 thế hệ kiến trúc: `Recall@10`, `Recall@20`, `NDCG@10`, `NDCG@20`, và mức tăng trưởng tương đối $\Delta\% = 100 \times \frac{\text{Metric}_{v7} - \text{Metric}_{\text{ref}}}{\text{Metric}_{\text{ref}}}$.
- Hiển thị song song qua PrettyTable và Pandas DataFrame.


In [ ]:
# Cell 9: Bảng Đối chuẩn Đa thế hệ Toàn diện (STAIR Baseline vs v4 vs v6 vs v7)
import pandas as pd
from prettytable import PrettyTable

table = PrettyTable()
table.field_names = ['Dataset', 'Kiến trúc / Phiên bản', *TRACKED_METRICS, 'Δ vs Baseline (%)', 'Δ vs v4 (%)', 'Δ vs v6 (%)']

all_results = {
    'sports': globals().get('metrics_sports', {}),
    'baby': globals().get('metrics_baby', {}),
    'electronics': globals().get('metrics_elec', {}),
}

df_rows = []

for ds in ('sports', 'baby', 'electronics'):
    ref_bl = BASELINE_REF[ds]['NDCG@20']
    ref_v4 = V4_REF[ds]['NDCG@20']
    ref_v6 = V6_REF[ds]['NDCG@20']

    # 1. Baseline
    table.add_row([ds, 'STAIR (baseline)', *[f"{BASELINE_REF[ds][m]:.4f}" for m in TRACKED_METRICS], '-', '-', '-'])
    df_rows.append({'Dataset': ds, 'Model': 'STAIR (baseline)', **{m: BASELINE_REF[ds][m] for m in TRACKED_METRICS}, 'Δ vs Baseline': 0.0, 'Δ vs v4': 0.0})

    # 2. v4
    d_v4_bl = 100 * (ref_v4 - ref_bl) / ref_bl
    table.add_row([ds, 'GD5-v4 (comparator)', *[f"{V4_REF[ds][m]:.4f}" for m in TRACKED_METRICS], f"{d_v4_bl:+.2f}", '-', '-'])
    df_rows.append({'Dataset': ds, 'Model': 'GD5-v4 (comparator)', **{m: V4_REF[ds][m] for m in TRACKED_METRICS}, 'Δ vs Baseline': d_v4_bl, 'Δ vs v4': 0.0})

    # 3. v6
    d_v6_bl = 100 * (ref_v6 - ref_bl) / ref_bl
    d_v6_v4 = 100 * (ref_v6 - ref_v4) / ref_v4
    table.add_row([ds, 'GD5-v6 (BCSR)', *[f"{V6_REF[ds][m]:.4f}" for m in TRACKED_METRICS], f"{d_v6_bl:+.2f}", f"{d_v6_v4:+.2f}", '-'])
    df_rows.append({'Dataset': ds, 'Model': 'GD5-v6 (BCSR)', **{m: V6_REF[ds][m] for m in TRACKED_METRICS}, 'Δ vs Baseline': d_v6_bl, 'Δ vs v4': d_v6_v4})

    # 4. v7
    cur_res = all_results.get(ds, {})
    cur_res_norm = {}
    if cur_res:
        for k, v in cur_res.items():
            for tm in TRACKED_METRICS:
                if k.lower() == tm.lower():
                    cur_res_norm[tm] = v
    if cur_res_norm and all(m in cur_res_norm for m in TRACKED_METRICS):
        d_bl = 100 * (cur_res_norm['NDCG@20'] - ref_bl) / ref_bl
        d_v4 = 100 * (cur_res_norm['NDCG@20'] - ref_v4) / ref_v4
        d_v6 = 100 * (cur_res_norm['NDCG@20'] - ref_v6) / ref_v6
        table.add_row([ds, 'STAIR5-v7 (UCR-D)', *[f"{cur_res_norm[m]:.4f}" for m in TRACKED_METRICS], f"{d_bl:+.2f}", f"{d_v4:+.2f}", f"{d_v6:+.2f}"] )
        df_rows.append({'Dataset': ds, 'Model': 'STAIR5-v7 (UCR-D)', **{m: cur_res_norm[m] for m in TRACKED_METRICS}, 'Δ vs Baseline': d_bl, 'Δ vs v4': d_v4})
    else:
        table.add_row([ds, 'STAIR5-v7 (UCR-D)', *['not measured'] * 4, '-', '-', '-'])

print(table)
results_df = pd.DataFrame(df_rows)
display(results_df) if 'display' in globals() else None


## Cell 10 📦 Đóng gói Lưu trữ Toàn bộ Minh chứng, Logs, Manifests & Checkpoints
- Nén toàn bộ kết quả, biểu đồ PNG, logs, telemetry và manifest JSON để nghiệm thu khóa luận tốt nghiệp.


In [ ]:
# Cell 10: Đóng gói Lưu trữ Toàn bộ Minh chứng (Logs, Checkpoints, Telemetry, Reports)
import os, shutil, json, time

archive_dir = '/kaggle/working/stair5_v7_artifacts'
os.makedirs(archive_dir, exist_ok=True)

# 1. Thu thập logs, reports và runs
target_dirs = [
    '/kaggle/working/logs/stair5_v7',
    '/kaggle/working/reports',
    '/kaggle/working/runs/stair5_v7',
]
for src in target_dirs:
    if os.path.exists(src):
        dst = os.path.join(archive_dir, os.path.basename(src))
        shutil.copytree(src, dst, dirs_exist_ok=True)

# 2. Tạo Manifest JSON tổng hợp
manifest = {
    "project": "STAIR5-v7 / UCR-D",
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "datasets": {
        "sports": {
            "test_metrics": globals().get("metrics_sports", {}),
            "best_epoch": globals().get("ep_sports", None),
            "train_time_min": (globals()["sports_time"] / 60) if "sports_time" in globals() else None,
        },
        "baby": {
            "test_metrics": globals().get("metrics_baby", {}),
            "best_epoch": globals().get("ep_baby", None),
            "train_time_min": (globals()["baby_time"] / 60) if "baby_time" in globals() else None,
        },
        "electronics": {
            "test_metrics": globals().get("metrics_elec", {}),
            "best_epoch": globals().get("ep_elec", None),
            "train_time_min": (globals()["electronics_time"] / 60) if "electronics_time" in globals() else None,
        },
    },
}

manifest_path = os.path.join(archive_dir, "stair5_v7_manifest.json")
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

# 3. Nén file zip hoàn chỉnh
zip_path = '/kaggle/working/stair5_v7_complete_artifacts.zip'
shutil.make_archive('/kaggle/working/stair5_v7_complete_artifacts', 'zip', archive_dir)

print("=" * 80)
print("📦 ĐÓNG GÓI HOÀN TẤT THÀNH CÔNG!")
print(f"  * Thư mục Artifacts : {archive_dir}")
if os.path.exists(zip_path):
    print(f"  * File Nén Tổng Hợp : {zip_path} ({os.path.getsize(zip_path)/(1024**2):.2f} MB)")
print("=" * 80)
